
# Point Cloud Processing Workflow

This notebook processes 3D PLY point cloud files through several automated steps:
1. **Modify PLY Files**: Corrects the headers of raw PLY files to remove unnecessary spaces.
2. **Rotate PLY Files**: Aligns the point clouds with the origin using specified marker positions.
3. **Merge Files**: Combines the rotated point clouds into a single file.
4. **Voxelize**: Downsamples the point cloud to reduce its size while preserving the structure.
5. **Smooth**: Refines the point cloud's appearance by averaging neighboring points.
6. **Segment with Pre-trained Model**: Applies a pre-trained model to classify and segment the point cloud.


## Mount google drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# Setting Input and Output Folders

*   Set input data folder from Raw_Data_Dir
*   Set output data folder from DATA_DIR

Example Raw files can be found in the fixed (original) data set uploaded to Figshare

In [ ]:
expNumber=1
DATA_DIR = '...'
Raw_Data_Dir="..."
!ls Raw_Data_Dir

27_20220523T042502_M.ply  27_20220523T042502_S.ply


##Required Libraries



In [ ]:
""" libraries to import plyfile, Plydata, os, numpy, struct"""
!pip install plyfile
!pip install open3d
from plyfile import PlyData
import os
import struct
import numpy as np
import open3d as o3d
import math
import cv2
from keras.models import Sequential,load_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.7/399.7 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.3 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


##Required Functions

In [ ]:
### required functions

def modify_ply_from_Phenospex(input_filename, output_filename):
  """ read Phenospex point cloud PLY file"""
  """ read header from input file, delete SPACE in header, write to output file in PLY format"""
  with open(input_filename, "br") as f:   #open input file
    s = f.read()
    end_spaces = s.find(b'\x65\x6E\x64\x5F\x68\x65\x61\x64\x65\x72\x0A')  #find end_heder
    start_spaces = s.find(b'\x76\x65\x72\x74\x65\x78\x5F\x69\x6E\x64\x65\x78\x0A') + 13  #find sequence of spaces
    s= s.replace(s[:end_spaces], s[:start_spaces])   #delete spaces
    f.close()
  with open(output_filename, "bw") as f:   #write modified content to output file
    f.write(s[::])
    f.close()
  return output_filename

def read_ply_header_obj_info(filename):
     """ read Phenospex point cloud PLY file, header"""
     """ return string array contains obj_info elements """
     """ input file in modified format: modify_ply_from_Phenospex, modify header to readable format"""
     assert(os.path.isfile(filename))
     with open(filename, 'rb') as f:
        plydata = PlyData.read(f)
        obj_info = plydata.obj_info
     return obj_info

def read_ply_vertex_all(filename):
    """ read Phenospex point cloud PLY file, all property """
    """ return array of vertices """
    """ input file in modified format: modify_ply_from_Phenospex, modify header to readable format"""
    assert(os.path.isfile(filename))
    with open(filename, 'rb') as f:
        plydata = PlyData.read(f)
        num_verts = plydata['vertex'].count
        vertices = np.zeros(shape=[num_verts, 18], dtype=np.float32)
        vertices[:,0] = plydata['vertex'].data['x']
        vertices[:,1] = plydata['vertex'].data['y']
        vertices[:,2] = plydata['vertex'].data['z']
        vertices[:,3] = plydata['vertex'].data['intensity']
        vertices[:,4] = plydata['vertex'].data['profile']
        vertices[:,5] = plydata['vertex'].data['x_pos']
        vertices[:,6] = plydata['vertex'].data['red']
        vertices[:,7] = plydata['vertex'].data['green']
        vertices[:,8] = plydata['vertex'].data['blue']
        vertices[:,9] = plydata['vertex'].data['nir']
        vertices[:,10] = plydata['vertex'].data['ndvi']
        vertices[:,11] = plydata['vertex'].data['wvl1']
        vertices[:,12] = plydata['vertex'].data['wvl2']
        vertices[:,13] = plydata['vertex'].data['wvl3']
        vertices[:,14] = plydata['vertex'].data['wvl4']
        vertices[:,15] = plydata['vertex'].data['wvl5']
        vertices[:,16] = plydata['vertex'].data['wvl6']
        vertices[:,17] = plydata['vertex'].data['wvl7']
    return vertices

def write_ply_from_vertex(output_filename, vertex, obj_info):
    with open(output_filename, 'bw') as f:
           # Write header  .ply file
          f.write(bytes('ply\n', 'utf-8'))
          f.write(bytes('format binary_little_endian 1.0\n', 'utf-8'))
          # Write header comment
          f.write(bytes('comment PLY data file generated by Phenospex PlantEye modified by KIT CZU\n', 'utf-8'))
          # Write header obj_info
          for obj in obj_info:
            f.write(bytes('obj_info ','utf-8'))
            f.write(bytes(obj,'utf-8'))
            f.write(bytes('\n', 'utf-8'))
          # Write header vertex
          f.write(bytes('element vertex ', 'utf-8'))
          f.write(bytes(str(len(vertex)), 'utf-8'))
          f.write(bytes('\n', 'utf-8'))
          # Write header property list
          f.write(bytes('property float x\n', 'utf-8'))
          f.write(bytes('property float y\n', 'utf-8'))
          f.write(bytes('property float z\n', 'utf-8'))
          f.write(bytes('property uchar intensity\n', 'utf-8'))
          f.write(bytes('property int profile\n', 'utf-8'))
          f.write(bytes('property int x_pos\n', 'utf-8'))
          f.write(bytes('property uchar red\n', 'utf-8'))
          f.write(bytes('property uchar green\n', 'utf-8'))
          f.write(bytes('property uchar blue\n', 'utf-8'))
          f.write(bytes('property uchar nir\n', 'utf-8'))
          f.write(bytes('property uchar ndvi\n', 'utf-8'))
          f.write(bytes('property ushort wvl1\n', 'utf-8'))
          f.write(bytes('property ushort wvl2\n', 'utf-8'))
          f.write(bytes('property ushort wvl3\n', 'utf-8'))
          f.write(bytes('property ushort wvl4\n', 'utf-8'))
          f.write(bytes('property ushort wvl5\n', 'utf-8'))
          f.write(bytes('property ushort wvl6\n', 'utf-8'))
          f.write(bytes('property ushort wvl7\n', 'utf-8'))
          #Write element face 0
          f.write(bytes('element face 0\n', 'utf-8'))
          f.write(bytes('property list uchar int vertex_index\n', 'utf-8'))
          f.write(bytes('end_header\n', 'utf-8'))
          #Write 3D points from vertex to .ply file
          for i in range (vertex.shape[0]):
           f.write(bytearray(struct.pack("f", vertex[i,0]))) # x coordinate
           f.write(bytearray(struct.pack("f", vertex[i,1]))) # y coordinate
           f.write(bytearray(struct.pack("f", vertex[i,2]))) # z coordinate
           f.write(bytearray(struct.pack("B", round(vertex[i,3])))) # intesity
           f.write(bytearray(struct.pack("i", round(vertex[i,4])))) # profile
           f.write(bytearray(struct.pack("i", round(vertex[i,5])))) # x_pos
           f.write(bytearray(struct.pack("B",round(vertex[i,6])))) #  r from RGB
           f.write(bytearray(struct.pack("B",round(vertex[i,7])))) #  g from RGB
           f.write(bytearray(struct.pack("B",round(vertex[i,8])))) #  b from RGB
           f.write(bytearray(struct.pack("B",round(vertex[i,9])))) # nir
           f.write(bytearray(struct.pack("B",round(vertex[i,10])))) # ndvi
           f.write(bytearray(struct.pack("H",round(vertex[i,11])))) # wvl1
           f.write(bytearray(struct.pack("H",round(vertex[i,12])))) # wvl2
           f.write(bytearray(struct.pack("H",round(vertex[i,13])))) # wvl3
           f.write(bytearray(struct.pack("H",round(vertex[i,14])))) # wvl4
           f.write(bytearray(struct.pack("H",round(vertex[i,15])))) # wvl5
           f.write(bytearray(struct.pack("H",round(vertex[i,16])))) # wvl6
           f.write(bytearray(struct.pack("H",round(vertex[i,17])))) # wvl7
    f.close()
    return output_filename

def rotate(origin, point, angle):
    """
    Rotate a point counterclockwise by a given angle around a given origin.

    The angle should be given in radians.
    """
    ox, oy = origin
    px, py = point

    qx = ox + math.cos(angle) * (px - ox) - math.sin(angle) * (py - oy)
    qy = oy + math.sin(angle) * (px - ox) + math.cos(angle) * (py - oy)
    return qx, qy

def convert_obj_info_2_dict(obj_info):
    fileInfo = {}
    for line in obj_info:
        words = line.split(" ")
        for j,word in enumerate(words):
            if word.find("marker_x_start")>-1:
                fileInfo["marker_x_start"]=float(words[j+1][:-1])
            elif word.find("marker_y_start")>-1:
                fileInfo["marker_y_start"]=float(words[j+1][:-1])
            elif word.find("marker_z_start")>-1:
                fileInfo["marker_z_start"]=float(words[j+1][:-1])

            elif word.find("marker_x_stop")>-1:
                fileInfo["marker_x_stop"]=float(words[j+1][:-1])
            elif word.find("marker_y_stop")>-1:
                fileInfo["marker_y_stop"]=float(words[j+1][:-1])
            elif word.find("marker_z_stop")>-1:
                fileInfo["marker_z_stop"]=float(words[j+1][:-1])

            elif word.find("field_y_origin") > -1:
                fileInfo["field_y_origin"] = float(words[j + 1][:-1])
            elif word.find("field_z_origin") > -1:
                fileInfo["field_z_origin"] = float(words[j + 1][:-1])
            elif word.find("field_x_origin") > -1:
                fileInfo["field_x_origin"] = float(words[j + 1][:-1])

            elif word.find("field_x_period") > -1:
                fileInfo["field_x_period"] = float(words[j + 1][:-1])
            elif word.find("field_y_period") > -1:
                fileInfo["field_y_period"] = float(words[j + 1][:-1])
            elif word.find("field_z_period") > -1:
                fileInfo["field_z_period"] = float(words[j + 1][:-1])

            elif word.find("ref_x_start") > -1:
                fileInfo["ref_x_start"] = float(words[j + 1][:-1])
            elif word.find("ref_y_start") > -1:
                fileInfo["ref_y_start"] = float(words[j + 1][:-1])
            elif word.find("ref_z_start") > -1:
                fileInfo["ref_z_start"] = float(words[j + 1][:-1])

            elif word.find("ref_x_stop") > -1:
                fileInfo["ref_x_stop"] = float(words[j + 1][:-1])
            elif word.find("ref_y_stop") > -1:
                fileInfo["ref_y_stop"] = float(words[j + 1][:-1])
            elif word.find("ref_z_stop") > -1:
                if words[j + 1].find("}") > -1:
                    fileInfo["ref_z_stop"] = float(words[j + 1][:-2])
                else:
                    fileInfo["ref_z_stop"] = float(words[j + 1][:-1])
            elif word.find("y_sectors") > -1:
                fileInfo["y_sectors"] = float(words[j + 1][:-1])

            elif word.find("pitch_pe") > -1:
                fileInfo["pitch_pe"] = float(words[j + 1][:-1])
    return fileInfo

def rotate2origin(input_filename, output_filename):
    header_obj_info = read_ply_header_obj_info(input_filename)
    fileGlobals=convert_obj_info_2_dict(header_obj_info)

    marker_x_start = fileGlobals["marker_x_start"]
    marker_x_stop = fileGlobals["marker_x_stop"]

    marker_z_start = fileGlobals["marker_z_start"]
    marker_z_stop = fileGlobals["marker_z_stop"]

    marker_x = (marker_x_start + marker_x_stop) / 2
    marker_z = (marker_z_start + marker_z_stop) / 2
    angle = fileGlobals["pitch_pe"]
    # verts = read_ply_vertex_all(input_filename)
    pcd = o3d.io.read_point_cloud(input_filename)
    verts=np.asarray(pcd.points)

    x, z = rotate([marker_x, marker_z], [verts[:, 0], verts[:, 2]], (-angle / 180) * math.pi)
    verts[:, 0] = x - marker_x
    verts[:, 2] = z - marker_z

    vectors =  o3d.utility.Vector3dVector(verts)
    pcd.points=vectors
    # write_ply_from_vertex(output_filename, verts, header_obj_info)
    o3d.io.write_point_cloud(output_filename, pcd)

def voxelization(pcd,voxelsize=2):
    pcd = pcd.voxel_down_sample(voxel_size=voxelsize)
    return pcd

def smoothing(pcd,adj_count=50):
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    for i in range(np.asarray(pcd.points).shape[0]):
        adj_index = pcd_tree.search_knn_vector_3d(pcd.points[i], adj_count)
        colors = np.mean(np.asarray(pcd.colors)[adj_index[1], :], axis=0)
        pcd.colors[i] = colors
    return pcd

def merge2ply(file1,file2):
    pcd1 = o3d.io.read_point_cloud(file1)
    pcd2 = o3d.io.read_point_cloud(file2)
    pcd1.points=o3d.utility.Vector3dVector(np.concatenate((np.asarray(pcd1.points),np.asarray(pcd2.points)),axis=0))
    pcd1.colors=o3d.utility.Vector3dVector(np.concatenate((np.asarray(pcd1.colors),np.asarray(pcd2.colors)),axis=0))
    return pcd1

def InverseCoordZ(input_filename, output_filename):
    pcd = o3d.io.read_point_cloud(input_filename)
    verts=np.asarray(pcd.points)
    verts[:, 2] =verts[:, 2]*-1
    vectors =  o3d.utility.Vector3dVector(verts)
    pcd.points = vectors
    o3d.io.write_point_cloud(output_filename, pcd)
    return pcd

def ply2png(input_filename, output_filename):
    pcd = o3d.io.read_point_cloud(input_filename)
    equ = cv2.equalizeHist((np.asarray(pcd.colors)*255).astype(np.uint8))
    pcd.colors = o3d.utility.Vector3dVector(equ.astype(np.float32)/255)
    vis = o3d.visualization.Visualizer()
    vis.create_window(width = 300, height=500,  visible=True)
    vis.create_window()
    vis.add_geometry(pcd)
    #o3d.visualization.ViewControl.set_zoom(vis.get_view_control(), 0.7)
    vis.capture_screen_image(output_filename[:-3]+".png",True)


def segment_Data(pcd,file):
    model = Sequential()
    model = load_model('/content/gdrive/Shareddrives/3D point clouds/NN_Models/all_species_NN_model.h5')

    X=np.concatenate((np.asarray(pcd.points),np.asarray(pcd.colors)*255),axis=1)
    X_org=X.copy()
    # X=scale.fit_transform(X)

    #X = X[:, :]
    Y = model.predict(X)
    #pdb.set_trace()
    bacg=X_org[Y[:,0]<=0.5]
    plants=X_org[Y[:,0]>0.5]
    "Inversion of Z coordinate"
    bacg[:, 2] = bacg[:, 2] * -1
    plants[:, 2] = plants[:, 2] * -1

    pcd_plants = o3d.geometry.PointCloud()
    pcd_plants.points = o3d.utility.Vector3dVector(plants[:, :3])
    pcd_plants.colors = o3d.utility.Vector3dVector(plants[:, 3:]/255)
    pcd_plants = pcd_plants.remove_statistical_outlier(nb_neighbors=100, std_ratio=2.0)[0]

    pcd_back = o3d.geometry.PointCloud()
    pcd_back.points = o3d.utility.Vector3dVector(bacg[:, :3])
    pcd_back.colors = o3d.utility.Vector3dVector(bacg[:, 3:]/255)

    if not os.path.exists(DATA_DIR+"6_segmentation_result_background/"):
        os.makedirs(DATA_DIR+"6_segmentation_result_background/")
    if not os.path.exists(DATA_DIR+"6_segmentation_result_plants/"):
        os.makedirs(DATA_DIR+"6_segmentation_result_plants/")


    if not os.path.exists(DATA_DIR+"7_partitioned_trays4plants/"+file[:-4]):
        os.makedirs(DATA_DIR+"7_partitioned_trays4plants/"+file[:-4])
    if not os.path.exists(DATA_DIR+"7_partitioned_trays4background/"+file[:-4]):
        os.makedirs(DATA_DIR+"7_partitioned_trays4background/"+file[:-4])

    print("veri sayısı: "+str(len(pcd_plants.points)))
    o3d.io.write_point_cloud(DATA_DIR+"6_segmentation_result_plants/"+file[:-4]+"_plants.pcd" , pcd_plants)
    o3d.io.write_point_cloud(DATA_DIR+"6_segmentation_result_background/"+file[:-4]+"_background.pcd" , pcd_back)

    "Partition of cleaned data"
    if os.path.exists(DATA_DIR + "1_modified/"+file[:-4]+"_M.ply"):
        header_obj_info = read_ply_header_obj_info((Raw_Data_Dir + file[:-4]+"_M.ply"))
    else:
        header_obj_info = read_ply_header_obj_info((Raw_Data_Dir + file[:-4]+"_S.ply"))

    fileGlobals=convert_obj_info_2_dict(header_obj_info)
    field_y_origin=fileGlobals["field_y_origin"]
    field_y_period=fileGlobals["field_y_period"]
    field_x_period=fileGlobals["field_x_period"]
    y_sectors=int(fileGlobals["y_sectors"])

    raw1=plants[plants[:,0]<=0]
    raw2=plants[plants[:,0]>0]
    raw1[:,0]=raw1[:,0]+field_x_period/2
    raw2[:,0]=raw2[:,0]-field_x_period/2

    coor_y=field_y_origin
    file_sp=file.split("_")
    for i in range(y_sectors):
        tray=np.copy(raw1[raw1[:,1]>coor_y])
        tray=tray[tray[:,1]<=(coor_y+field_y_period)]
        tray[:,1]=tray[:,1]-(coor_y+field_y_period/2)

        part_plant = o3d.geometry.PointCloud()
        part_plant.points = o3d.utility.Vector3dVector(tray[:, :3])
        part_plant.colors = o3d.utility.Vector3dVector(tray[:, 3:] / 255)
        o3d.io.write_point_cloud(DATA_DIR + "7_partitioned_trays4plants/" + file[:-4] +"/"+str(expNumber)+"_"+file_sp[0]+"_0_"+str(i)+"_"+file_sp[1]+".pcd", part_plant)

        coor_y=coor_y+field_y_period


    coor_y=field_y_origin
    for i in range(y_sectors):
        tray=np.copy(raw2[raw2[:,1]>coor_y])
        tray=tray[tray[:,1]<=(coor_y+field_y_period)]
        tray[:,1]=tray[:,1]-(coor_y+field_y_period/2)

        part_back = o3d.geometry.PointCloud()
        part_back.points = o3d.utility.Vector3dVector(tray[:, :3])
        part_back.colors = o3d.utility.Vector3dVector(tray[:, 3:] / 255)
        o3d.io.write_point_cloud(DATA_DIR + "7_partitioned_trays4plants/" + file[:-4] +"/"+str(expNumber)+"_"+file_sp[0]+"_1_"+str(i)+"_"+file_sp[1]+".pcd", part_back)
        coor_y=coor_y+field_y_period



    print("---Segmentation process completed --- check Data/Data2Clean/6_segmentation_result_plants/")


#Calling Functions for Modify Data and Rotate Data

In [ ]:
#modify
plyfile_directory = os.listdir(Raw_Data_Dir)
for i, plyfile in enumerate(plyfile_directory):
    modify_ply_from_Phenospex(Raw_Data_Dir + plyfile, Raw_Data_Dir + plyfile)
print ("***Data Modified***")

#rotate
outputFolder = DATA_DIR + "/modified_rotated/"
if not os.path.exists(outputFolder):
    os.makedirs(outputFolder)
plyfile_directory = os.listdir(Raw_Data_Dir)
for i, plyfile in enumerate(plyfile_directory):
    rotated_data=rotate2origin(Raw_Data_Dir + plyfile, outputFolder +  plyfile)
print ("***Rotated***")






***Data Modified***
***Rotated***


#Calling Functions for Merging Data, Voxelizing Data, Smoothing Data, and Segmenting Point Cloud Data

In [ ]:
#merge
plyfile_directory = os.listdir(DATA_DIR + "/modified_rotated/")
file_list=[]
for i in range(len(plyfile_directory)):
    file_list.append(plyfile_directory[i])
file_list2=file_list.copy()
list_done=[]
for file1 in file_list:
    if list_done.count(file1)== 1:
        continue
    for temp in file_list2:
        if temp.__contains__(file1[:-10]) and temp!=file1 :
            file2=temp
            list_done.append(file1)
            list_done.append(file2)
            break

    merged_data= merge2ply(DATA_DIR + "/modified_rotated/"+ file1, DATA_DIR + "/modified_rotated/" + file2)
    print ("***Merged***")
    print(merged_data.points)

    #voxelize
    voxelized_data= voxelization(merged_data)
    print ("***Voxelized***")
    print(voxelized_data.points)

    #smothing
    smothed_data=smoothing(voxelized_data)
    #print ("***Smothed***")

    #segmenting
    segment_Data(smothed_data, file1[:-6]+".ply")


***Merged***
std::vector<Eigen::Vector3d> with 13834647 elements.
Use numpy.asarray() to access data.
***Voxelized***
std::vector<Eigen::Vector3d> with 2429174 elements.
Use numpy.asarray() to access data.


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


75912/75912 ━━━━━━━━━━━━━━━━━━━━ 106s 1ms/step
veri sayısı: 135536
---Segmentation process completed --- check Data/Data2Clean/6_segmentation_result_plants/
